# Eval Overview - Gene Expression Prediction

Our first eval category is predicting gene expression. This eval runs on top of the predictions produced by the [linear expression decoder](https://github.com/GPTomics/biojepa/blob/main/layer_explainers/explainer_eval_decoders.ipynb). Since our model is trained on perturbSeq, which is gene expression data, this is one of the more natural evals for us to perform. Additionally, there are many models in the space working on perturbation prediction so it's a great eval for comparative performance.

Our eval suite does 3 levels of analysis on the expression prediction: sample-level, per-perturbation, and cross-perturbation. We perform these on the full test set, on test sets subset by dataset, and on test sets subset by cell type. As you walk through the notebook you'll see that we evaluate on multiple dimensions to ensure we have a thorough understanding on where our model is working well and where it's struggling.

In [1]:
import numpy as np
from scipy.stats import pearsonr, spearmanr
from sklearn.metrics import r2_score
import torch

In [2]:
SEED = 1337
np.random.seed(SEED)

## Data Prep
We'll start by preparing our data. For this evaluation, we need to have the true cell expressions, the control expression for each cell, and the perturbation information. To show both sample level and perturbation level we'll mock up 6 samples across 4 perturbations. A "perturbation" is a unique combination of a sequence, target, modality, and mode applied to a cell type. A perturbation can span datasets. For the 6 samples we'll show the real control cell expression, predicted control cell expression (running the student encoder and then linear decoder), real case cell expression, predicted case cell expression, and the perturbation. We'll go ahead and do all 6 samples in batches.

For predicted expression, we use the data created by the [linear expression decoder](https://github.com/GPTomics/biojepa/blob/main/layer_explainers/explainer_eval_decoders.ipynb). As a reminder, the decoder takes the mean ($\mu$) BioJEPA-AC output based on the perturbations and cell expression pattern, and then uses a linear layer to project down to a $[\text{n\_genes},\text{1}]$ matrix with a single value per gene representing the expression.

We'll stage data to show a few different predictions: a strong prediction, weak prediction, inverse prediction, and over prediction.

In [3]:
unique_perts = 4
num_genes = 8
num_cells = 6
TOP_K = 3

**Perturbations**

We'll first start with our perturbations. Even though we have 6 cells, our sample will only have 4 unique perturbations. To define a unique perturbation, it's not just about what we target, but the context of it. Because of this, we represent a unique perturbation as $\text{(seq id, targ id, modality id, mode id, cell type)}$. IDs are used since our model keeps the perturbation information in separate caches from our sample expression counts to avoid heavy duplication of information.

We'll also create a mapping of the sample to the perturbation, where the first two samples map to the first perturbation, the next two samples to the second perturbation, and then the final two samples each have a unique perturbation.

In [4]:
pert_keys = [                                                                      
      (0, 0, 0, 0, 0),  # pert 0: DNA CRISPRi, cell type 0 
      (1, 1, 0, 0, 0),  # pert 1: DNA CRISPRi, cell type 0 
      (2, 2, 0, 1, 0),  # pert 2: DNA CRISPRa, cell type 0 
      (3, 3, 2, 4, 0),  # pert 3: Chemical inhibitor, cell type 0
]


sample_to_pert = [0, 0, 1, 1, 2, 3]

**Control Cells**

Next we'll show the control cell values. Recall that for our inference we pair together a perturbed cell with a random control cell from the same batch. This allows us to take an approximate change in prediction. Beyond just the control cell expression, to calculate our predicted change in expression, we run the expression predictor on the control cell's latent representation `z_context`. We'll discuss the calculation more but to show this we'll also have the predicted control expression.

For the predicted, we'll show a minor shift to highlight that often the prediction is not perfect.

In [5]:
real_control = np.array([
    [2.1, 3.4, 1.2, 4.1, 2.6, 3.1, 1.4, 4.6], 
    [1.9, 3.6, 0.8, 3.9, 2.4, 2.9, 1.6, 4.4], 
    [2.0, 3.3, 1.1, 4.2, 2.3, 3.2, 1.3, 4.3], 
    [2.2, 3.7, 0.9, 3.8, 2.7, 2.8, 1.7, 4.7], 
    [2.0, 3.5, 1.0, 4.0, 2.5, 3.0, 1.5, 4.5], 
    [2.0, 3.5, 1.0, 4.0, 2.5, 3.0, 1.5, 4.5], 
])
pred_control = np.array([
    [2.0, 3.3, 1.3, 4.0, 2.5, 3.2, 1.3, 4.5], 
    [1.8, 3.5, 0.9, 3.8, 2.3, 3.0, 1.5, 4.3], 
    [1.9, 3.2, 1.2, 4.1, 2.2, 3.3, 1.2, 4.2], 
    [2.1, 3.6, 1.0, 3.7, 2.6, 2.9, 1.6, 4.6], 
    [1.9, 3.4, 1.1, 3.9, 2.4, 3.1, 1.4, 4.4], 
    [1.9, 3.4, 1.1, 3.9, 2.4, 3.1, 1.4, 4.4], 
])
real_control.shape, pred_control.shape

((6, 8), (6, 8))

**Case Cell**

Next we'll show the case cell values. In our raw data we have the real expression of the perturbed cell. We pair this together with the linear expression decoder output based on the mean prediction, $\mu$, from the ACPredictor output. The mean prediction is based on the control cell latent representation `z_context` and the perturbations. You'll quickly be able to see that there is a difference between the real values and the predicted values. We've staged the data so that the first two samples predict closely, the next two samples weakly, the fifth sample predicts in the wrong direction, and the final sample over predicts. You'll see how these calculations flow through.

In [6]:
real_case = np.array([
    [2.8, 2.3, 1.6, 6.0, 2.0, 4.7, 1.2, 2.9], 
    [2.8, 2.2, 1.0, 6.0, 2.0, 4.3, 1.6, 2.5], 
    [2.1, 3.2, 1.2, 4.1, 2.4, 3.3, 1.3, 4.4], 
    [2.3, 3.6, 0.9, 3.7, 2.7, 2.8, 1.7, 4.7], 
    [2.5, 2.7, 1.6, 3.0, 2.8, 3.7, 1.1, 5.7], 
    [2.3, 3.1, 1.2, 4.5, 2.2, 3.6, 1.4, 4.9], 
])

pred_case = np.array([
    [2.6, 2.3, 1.8, 5.8, 2.0, 4.6, 1.0, 3.0],  # strong
    [2.6, 2.3, 1.2, 5.8, 2.0, 4.5, 1.4, 2.6],  # strong
    [2.05, 3.05, 1.25, 4.05, 2.3, 3.35, 1.15, 4.3],  # weak
    [2.15, 3.6, 0.95, 3.6, 2.65, 2.95, 1.6, 4.65],  # weak
    [1.6, 3.9, 1.6, 4.5, 2.2, 3.9, 1.7, 3.9],  # inverse
    [2.8, 2.2, 1.7, 5.4, 1.5, 4.9, 1.1, 5.6],  # over
])

real_case.shape, pred_case.shape

((6, 8), (6, 8))

## Calculate Delta

A major component of our expression benchmark is not looking at absolute predictions, but the change in expression. Some claim that this simplifies the task. Biologically, we see this as addressing the important questions: can you predict what will change, in what direction, and by how much. We focus on calculating two sample level differences:
1. `pred_delta` - the predicted change in expression as calculated by $\hat{\delta}_g = \hat{x}^{\text{case}}_g - \hat{x}^{\text{ctrl}}_g$. This value compares the predicted perturbed expression (`pred_case`) from the predicted control expression (`pred_control`). We use the predicted control expression to isolate BioJEPA-AC's learned perturbation effect from any baseline reconstruction error.
2. `real_delta` - the real change in expression as calculated by $\delta_g = x^{\text{case}}_g - x^{\text{ctrl}}_g$. This is our source of truth.

With this calculation you'll see how we end up seeing both increases and decreases in expression by gene. We'll end up comparing these by different slices in our calculations.

In [7]:
pred_delta = pred_case - pred_control

pred_delta.shape, pred_delta

((6, 8),
 array([[ 0.6 , -1.  ,  0.5 ,  1.8 , -0.5 ,  1.4 , -0.3 , -1.5 ],
        [ 0.8 , -1.2 ,  0.3 ,  2.  , -0.3 ,  1.5 , -0.1 , -1.7 ],
        [ 0.15, -0.15,  0.05, -0.05,  0.1 ,  0.05, -0.05,  0.1 ],
        [ 0.05,  0.  , -0.05, -0.1 ,  0.05,  0.05,  0.  ,  0.05],
        [-0.3 ,  0.5 ,  0.5 ,  0.6 , -0.2 ,  0.8 ,  0.3 , -0.5 ],
        [ 0.9 , -1.2 ,  0.6 ,  1.5 , -0.9 ,  1.8 , -0.3 ,  1.2 ]]))

In [8]:
real_delta = real_case - real_control

real_delta.shape, real_delta

((6, 8),
 array([[ 0.7, -1.1,  0.4,  1.9, -0.6,  1.6, -0.2, -1.7],
        [ 0.9, -1.4,  0.2,  2.1, -0.4,  1.4,  0. , -1.9],
        [ 0.1, -0.1,  0.1, -0.1,  0.1,  0.1,  0. ,  0.1],
        [ 0.1, -0.1,  0. , -0.1,  0. ,  0. ,  0. ,  0. ],
        [ 0.5, -0.8,  0.6, -1. ,  0.3,  0.7, -0.4,  1.2],
        [ 0.3, -0.4,  0.2,  0.5, -0.3,  0.6, -0.1,  0.4]]))

## Calculate Absolutes

You might look at what our model predicted and think "great we have the predicted changes in expression, and we have absolute expression, we're ready to start calculating." This is a bit short sighted though as our absolute predicted expression is based on the model's understanding of the predicted control expression. This fallacy is **the** reason why we focus on predicted changes over just absolute numbers. If the model can predict the change in expression perfect but just has a different expectation for what the control is, we can have a big issue in our absolute numbers and think we have a crap model when we actually have a great model.

Because of this we actually create our predicted absolute expression based on the true control and the predicted delta instead of the predicted absolute. We result in calculating the `pred_abs` as $\hat{x}^{\text{abs}}_g = x^{\text{ctrl}}_g + \hat{\delta}_g$.

Overall this gives us a way to really evaluate if our model has learned how to shift cells based on perturbation. There is one downside: if we have a very poor prediction, we can end up with a negative gene expression. While we could handle this as just meaning we're very poor at predicting, we will use a minor bandaid here and just say: the minimum expression we can have is 0. We do this by applying a clamp in our evals but, since this is just numpy, we'll use a slightly different function. While this will hide very poor negative predictions, we're willing to take this risk with our bandaid. You'll see with our sample data that this isn't an issue since we have all values above zero (on purpose). We'll also create `real_abs` which is just a copy of the case cell expressions.

In [9]:
pred_abs = np.maximum(real_control + pred_delta, 0.0)
pred_abs.shape, pred_abs

((6, 8),
 array([[2.7 , 2.4 , 1.7 , 5.9 , 2.1 , 4.5 , 1.1 , 3.1 ],
        [2.7 , 2.4 , 1.1 , 5.9 , 2.1 , 4.4 , 1.5 , 2.7 ],
        [2.15, 3.15, 1.15, 4.15, 2.4 , 3.25, 1.25, 4.4 ],
        [2.25, 3.7 , 0.85, 3.7 , 2.75, 2.85, 1.7 , 4.75],
        [1.7 , 4.  , 1.5 , 4.6 , 2.3 , 3.8 , 1.8 , 4.  ],
        [2.9 , 2.3 , 1.6 , 5.5 , 1.6 , 4.8 , 1.2 , 5.7 ]]))

In [10]:
real_abs = real_case.copy()
real_abs.shape, real_abs

((6, 8),
 array([[2.8, 2.3, 1.6, 6. , 2. , 4.7, 1.2, 2.9],
        [2.8, 2.2, 1. , 6. , 2. , 4.3, 1.6, 2.5],
        [2.1, 3.2, 1.2, 4.1, 2.4, 3.3, 1.3, 4.4],
        [2.3, 3.6, 0.9, 3.7, 2.7, 2.8, 1.7, 4.7],
        [2.5, 2.7, 1.6, 3. , 2.8, 3.7, 1.1, 5.7],
        [2.3, 3.1, 1.2, 4.5, 2.2, 3.6, 1.4, 4.9]]))

## Sample Level

While we have sample level information, the volume of it can create noisy evals. Because of this we only run a few metrics at the sample level and, the metrics we run, focus on evaluating the expression change metrics (delta). The two we run are mean-squared-error and the Pearson's correlation coefficient. MSE is run across all our data while Pearson's $r$ is calculated on only the top 20 differentially expressed genes per sample.

We'll dive into each calculation.

### Mean Squared Error (MSE)

Our first calculation will evaluate simply how far off are our expression change predictions from the real changes. To do this we calculate the mean squared error as follows:

$$\text{MSE}_{\text{sample}}=\frac{1}{N}\sum_{i=1}^{N}\frac{1}{G}\sum_{g=1}^{G}(\hat{\delta}_{i,g} - \delta_{i,g})^2$$

If you look at the formula, you can see that this calculation will quickly be dominated by the largest expression changes. This is a large reason why we log normalize expression counts, that way large order of magnitude changes do not dominate our optimization. This is especially important since in many of our perturbation cases a vast majority of our genes will have very minor changes and we need to make sure we're able to predict that just as well as large changes.

After we calculate per sample MSE, we then take a final mean to get the dataset level mean MSE. We can see that because of how we seeded our data, the inverse prediction (cell 4) and over prediction (cell 5) have large MSE while the strong and weak predictions have small MSE. Also note that all MSE are positive so they more indicate the order of magnitude of the error, and not the direction.

In [11]:
per_sample_mse = np.mean((pred_delta - real_delta)**2, axis=1)
per_sample_mse.shape, per_sample_mse

((6,),
 array([0.0175   , 0.0175   , 0.001875 , 0.0028125, 1.0675   , 0.58     ]))

In [12]:
sample_mse = np.mean(per_sample_mse)
sample_mse

np.float64(0.28119791666666666)

### Top 20 DEG Pearson's $r$

We next calculate the Pearson's correlation coefficient, $r$, per sample. Pearson's $r$ measures how linearly correlated the predicted and real expression delta profiles are across genes for each sample, regardless of scale. The value of this eval is that if our expression is very linearly correlated, but the values are off, our model is still useful as it's learned the profile and we just need to find the right scaling factor. For our sample level Pearson's, since this is a benchmark that we compare against other models, we calculate per sample but restrict to the top 20 differentially expressed genes. We pull out the largest (by absolute value) top 20 gene expression changes from our real values `real_delta`, and then compare those against the calculated changes for those same genes and then take the dataset mean. The resulting calculation is:

$$r_{\text{sample}} = \frac{1}{N}\sum_{i=1}^{N}\frac{\sum_{k=1}^{K}(\hat{\delta}_{i,k} - \bar{\hat{\delta}}_i)(\delta_{i,k} - \bar{\delta}_i)}{\sqrt{\sum_{k=1}^{K}(\hat{\delta}_{i,k} - \bar{\hat{\delta}}_i)^{2}} \cdot \sqrt{\sum_{k=1}^{K}(\delta_{i,k} - \bar{\delta}_i)^{2}}}$$

where $K=20$ genes are selected per sample as the largest $|\delta_{i,g}|$. For this example, since we only have 8 genes, instead of taking the top 20, we'll show it by taking only the top 3.

In [13]:
per_sample_corr = []
for i in range(num_cells):
    print(f'----CELL {i}----')
    top_20_idx = np.argsort(np.abs(real_delta[i]))[-TOP_K:]
    pred_top, real_top = pred_delta[i][top_20_idx], real_delta[i][top_20_idx]
    print(f'Top {TOP_K} expression deltas: pred {pred_top} | real {real_top}')
    corr, _ = pearsonr(pred_top, real_top)
    p_corr = 0.0 if np.isnan(corr) else float(corr)
    print(f'Pearson\'s r {p_corr}')
    per_sample_corr.append(p_corr)

per_sample_corr = np.array(per_sample_corr)
per_sample_corr.shape, per_sample_corr

----CELL 0----
Top 3 expression deltas: pred [ 1.4 -1.5  1.8] | real [ 1.6 -1.7  1.9]
Pearson's r 0.9993477847421193
----CELL 1----
Top 3 expression deltas: pred [ 1.5 -1.7  2. ] | real [ 1.4 -1.9  2.1]
Pearson's r 0.9992110001785024
----CELL 2----
Top 3 expression deltas: pred [ 0.1  -0.05  0.1 ] | real [ 0.1 -0.1  0.1]
Pearson's r 1.0
----CELL 3----
Top 3 expression deltas: pred [ 0.05 -0.1   0.  ] | real [ 0.1 -0.1 -0.1]
Pearson's r 0.7559289460184526
----CELL 4----
Top 3 expression deltas: pred [ 0.5  0.6 -0.5] | real [-0.8 -1.   1.2]
Pearson's r -1.0
----CELL 5----
Top 3 expression deltas: pred [1.2 1.5 1.8] | real [0.4 0.5 0.6]
Pearson's r 1.0


((6,),
 array([ 0.99934778,  0.999211  ,  1.        ,  0.75592895, -1.        ,
         1.        ]))

In [14]:
sample_corr = np.mean(per_sample_corr)
sample_corr

np.float64(0.6257479551565125)

## Perturbation Level

Our next set of evals is done at the perturbation level. As a reminder, we consider a perturbation to be unique if it's the same sequence, target, mode, modality applied to the same cell type. To analyze per perturbation, we review all of the expression data we have per perturbation and then take the mean to get a single value per perturbation, giving us per gene expression data for each perturbation.

Once we have per perturbation values, we can calculate our evaluation. We run a number of per perturbation evaluations including MSE, $R^2$ correlation, and Pearson's $r$. We run $R^2$ and Pearson's $r$ on both all the genes and the top 50 DEGs. We use "top 50" to align with common industry benchmarks in other papers.

We'll first aggregate our data.

### Aggregate Data Per Perturbation

We'll first create our per-perturbation data. We'll use our `sample_to_pert` to help identify which perturbation each sample belongs to. We'll iterate through our perturbations, find which samples belong to the perturbation, pluck out the expression data for the samples, and then run a mean across them to get a single value per perturbation.

In [15]:
mean_pert_pred_delta = np.zeros((unique_perts, num_genes)) 
mean_pert_real_delta = np.zeros((unique_perts, num_genes)) 
mean_pert_pred_abs = np.zeros((unique_perts, num_genes)) 
mean_pert_real_abs = np.zeros((unique_perts, num_genes)) 
mean_pert_real_control = np.zeros((unique_perts, num_genes)) 

In [16]:
for pert_idx in range(unique_perts):                                            
    mask = [i for i, p in enumerate(sample_to_pert) if p == pert_idx]           
    mean_pert_pred_delta[pert_idx] = pred_delta[mask].mean(axis=0)                   
    mean_pert_real_delta[pert_idx] = real_delta[mask].mean(axis=0)
    mean_pert_pred_abs[pert_idx] = pred_abs[mask].mean(axis=0)
    mean_pert_real_abs[pert_idx] = real_abs[mask].mean(axis=0)
    mean_pert_real_control[pert_idx] = real_control[mask].mean(axis=0)

mean_pert_pred_delta.shape, mean_pert_pred_delta, mean_pert_real_delta, mean_pert_pred_abs, mean_pert_real_abs, mean_pert_real_control

((4, 8),
 array([[ 0.7  , -1.1  ,  0.4  ,  1.9  , -0.4  ,  1.45 , -0.2  , -1.6  ],
        [ 0.1  , -0.075,  0.   , -0.075,  0.075,  0.05 , -0.025,  0.075],
        [-0.3  ,  0.5  ,  0.5  ,  0.6  , -0.2  ,  0.8  ,  0.3  , -0.5  ],
        [ 0.9  , -1.2  ,  0.6  ,  1.5  , -0.9  ,  1.8  , -0.3  ,  1.2  ]]),
 array([[ 0.8 , -1.25,  0.3 ,  2.  , -0.5 ,  1.5 , -0.1 , -1.8 ],
        [ 0.1 , -0.1 ,  0.05, -0.1 ,  0.05,  0.05,  0.  ,  0.05],
        [ 0.5 , -0.8 ,  0.6 , -1.  ,  0.3 ,  0.7 , -0.4 ,  1.2 ],
        [ 0.3 , -0.4 ,  0.2 ,  0.5 , -0.3 ,  0.6 , -0.1 ,  0.4 ]]),
 array([[2.7  , 2.4  , 1.4  , 5.9  , 2.1  , 4.45 , 1.3  , 2.9  ],
        [2.2  , 3.425, 1.   , 3.925, 2.575, 3.05 , 1.475, 4.575],
        [1.7  , 4.   , 1.5  , 4.6  , 2.3  , 3.8  , 1.8  , 4.   ],
        [2.9  , 2.3  , 1.6  , 5.5  , 1.6  , 4.8  , 1.2  , 5.7  ]]),
 array([[2.8 , 2.25, 1.3 , 6.  , 2.  , 4.5 , 1.4 , 2.7 ],
        [2.2 , 3.4 , 1.05, 3.9 , 2.55, 3.05, 1.5 , 4.55],
        [2.5 , 2.7 , 1.6 , 3.  , 2.8 , 3.7 , 

### Coefficient of determination $R^2$
Our first calculation is the coefficient of determination, or $R^2$ between our real and predicted absolute expression values. This metric measures how much variance in the real expression our predictions explain by running the following:

$$R^{2}_\text{pert} = 1 - \frac{\sum_{g=1}^{G}(\hat{x}^{\text{abs}}_{p,g} - x^{\text{abs}}_{p,g})^{2}}{\sum_{g=1}^{G}(x^{\text{abs}}_{p,g} - \bar{x}^{\text{abs}}_p)^{2}}$$

where $\bar{x}^{\text{abs}}_p = \frac{1}{G}\sum_{g=1}^{G}x^{\text{abs}}_{p,g}$. Once we have the per-perturbation $R^2$ we then take the mean and median and report on it. By showing the median we help identify if there's a skew in our dataset. In our real dataset, we expect the median should be above the mean (remember we're comparing $R^2$) because, a lot of genes do not move and, if our model learned that correctly, we should have a large amount of high correlations and only a few lower correlations. If we see the median below the mean, we see that we're getting a skew right where a larger amount of correlations are below the mean with a few good outliers (aka bad training).

You might be wondering: why does $R^2$ use absolute expression values instead of deltas? Absolute expression captures both the baseline expression pattern and the perturbation effect, asking "can you reconstruct the overall expression profile." If we calculated on deltas instead, we'd only be asking "can you reconstruct the perturbation effect," which we evaluate separately with other metrics.

We run our $R^2$ calculation two different ways, on all our genes and on the top 50 DEG. For the top 50 DEG, we calculate the DEG per perturbation, taking from the real expression changes the genes with the top 50 expression changes based on absolute value.

**$R^2$ all genes**

We'll first start by calculating the $R^2$ on all genes for our absolute expression values. We'll iterate through each perturbation and calculate the $R^2$. After that we'll calculate the mean and median for our eval.

We can quickly see the correlation between our predicted and actual values for our first two perturbations, but that drops off for the last two. Also we'll see that we do indeed get the left-skew we want showing that most of our genes are correlated better than average. This is expected based on how we seeded the data.

In [17]:
per_pert_r2_all = []
for i in range(unique_perts):
    per_pert_r2_all.append(r2_score(mean_pert_real_abs[i], mean_pert_pred_abs[i]))
per_pert_r2_all = np.array(per_pert_r2_all)
per_pert_r2_all.shape, per_pert_r2_all

((4,), array([0.99370025, 0.9994298 , 0.37338347, 0.64525994]))

In [18]:
pert_r2_all_mean = np.mean(per_pert_r2_all)
pert_r2_all_median = np.median(per_pert_r2_all)   

pert_r2_all_mean, pert_r2_all_median

(np.float64(0.7529433654580222), np.float64(0.8194800935309179))

**$R^2$ top 50 DEG**

Next we'll again calculate the $R^2$ but this time only for the top differentially expressed genes (DEGs). This will eliminate most of the housekeeping genes from our calculation and really focus on genes that are shifting for our perturbation. We picked 50 here to better compare against other models. To get the top DEGs, we review the actual expression deltas and take the top 50 with the highest absolute value (for our example we'll take the top 3 again).

We'll cycle through each perturbation and slice out the top DEGs. Once we have those we'll calculate the $R^2$ based again on the absolute expression values. After that we'll calculate the mean and median for our eval.

We can quickly see the correlation between our predicted and actual values for our first two perturbations, but we see very strong negative correlations for the second two perts starting to highlight how poorly it was predicted. The `-2` $R^2$ for our fourth perturbation happens because the model may have gotten the direction right but overestimates magnitudes by ~3x, an issue magnified by our top-K selection.

In [19]:
per_pert_r2_topk = []
for i in range(unique_perts):
    print(f'----Pert {i}----')
    mp_r, mp_p  = mean_pert_real_abs[i], mean_pert_pred_abs[i]
    tk_idx = np.argsort(np.abs(mean_pert_real_delta[i]))[-TOP_K:]
    print(f'Top {TOP_K} expression deltas: real {mp_r[tk_idx]} | pred {mp_p[tk_idx]}')
    per_pert_r2_topk.append(r2_score(mp_r[tk_idx], mp_p[tk_idx]))
per_pert_r2_topk = np.array(per_pert_r2_topk)
per_pert_r2_topk.shape, per_pert_r2_topk

----Pert 0----
Top 3 expression deltas: real [4.5 2.7 6. ] | pred [4.45 2.9  5.9 ]
----Pert 1----
Top 3 expression deltas: real [2.2 3.4 3.9] | pred [2.2   3.425 3.925]
----Pert 2----
Top 3 expression deltas: real [2.7 3.  5.7] | pred [4.  4.6 4. ]
----Pert 3----
Top 3 expression deltas: real [4.9 4.5 3.6] | pred [5.7 5.5 4.8]


((4,), array([ 0.99038462,  0.99918122, -0.30769231, -2.47368421]))

In [20]:
pert_r2_topk_mean = np.mean(per_pert_r2_topk)
pert_r2_topk_median = np.median(per_pert_r2_topk)   

pert_r2_topk_mean, pert_r2_topk_median

(np.float64(-0.19795267003164574), np.float64(0.34134615384615397))

### Mean Squared Error (MSE)

Our next calculation will evaluate simply how far off are our expression change predictions from the real changes for a perturbation. This is the same calculation we had per sample, but in this case we'll run it across our perturbation aggregates.

In [21]:
per_pert_mse = np.mean((mean_pert_pred_delta - mean_pert_real_delta)**2, axis=1)
per_pert_mse.shape, per_pert_mse

((4,), array([1.43750e-02, 7.03125e-04, 1.06750e+00, 5.80000e-01]))

In [22]:
pert_mse_mean = np.mean(per_pert_mse)
pert_mse_median = np.median(per_pert_mse)

pert_mse_mean, pert_mse_median

(np.float64(0.41564453125), np.float64(0.29718749999999994))

### Pearson's $r$ All Genes
Next we'll calculate Pearson's correlation coefficient, $r$, per perturbation, both on our total dataset of genes and our top 50 DEGs. We'll apply a similar formula as we did per sample, but in this case per perturbation. We'll calculate both on all genes (on absolute and deltas) and our top 50 DEGs (on deltas). Pearson's on absolute expression measures how well the predicted profile matches the real one across all genes (dominated by the baseline expression pattern), while Pearson's on deltas isolates whether the model captured the perturbation effect specifically. When absolute Pearson's is high but delta Pearson's is low, the model is riding the control expression pattern but failing to predict the actual perturbation-induced changes well.

**All Genes**
We'll first start with all genes and calculate both on the absolute expression values and the expression deltas. You can see the value of having both by looking at how the all gene correlation is much higher than the delta. Additionally if you compare with our other per perturbation correlation you can see the value of different types of calculations. For example, we saw the last perturbation had a terrible $R^2$ but when we look at the Pearson's on the expression delta, it's a perfect 1.0. This is because the predicted expression for that perturbation predicts the right direction and relative proportions for every gene, just at ~3x the magnitude. Pearson's is scale-invariant, so a perfect linear scaling doesn't affect it. Additionally you'll see for the third perturbation we have a negative Pearson's on the expression delta because the model predicts the wrong direction on several genes, meaning as real deltas go up the predicted deltas go down. This shows the importance of more than 1 metric.

In [23]:
per_pert_pr_abs = []
per_pert_pr_delt = []
for i in range(unique_perts):
    p_abs, r_abs = mean_pert_pred_abs[i], mean_pert_real_abs[i]
    abs_corr, _ = pearsonr(p_abs, r_abs)
    abs_pcorr = 0.0 if np.isnan(abs_corr) else float(abs_corr)
    per_pert_pr_abs.append(abs_pcorr)
    
    p_delt, r_delt = mean_pert_pred_delta[i], mean_pert_real_delta[i]
    delt_corr, _ = pearsonr(p_delt, r_delt)
    delt_pcorr = 0.0 if np.isnan(delt_corr) else float(delt_corr)
    per_pert_pr_delt.append(delt_pcorr)
    
per_pert_pr_abs = np.array(per_pert_pr_abs)
per_pert_pr_delt = np.array(per_pert_pr_delt)

per_pert_pr_abs.shape, per_pert_pr_abs, per_pert_pr_delt.shape, per_pert_pr_delt

((4,),
 array([0.99738511, 0.9999286 , 0.6594819 , 0.93683002]),
 (4,),
 array([ 0.99820229,  0.92568728, -0.49692585,  1.        ]))

In [24]:
pert_abs_pcorr_mean = np.mean(per_pert_pr_abs)
pert_abs_pcorr_median = np.median(per_pert_pr_abs)
pert_abs_pcorr_mean, pert_abs_pcorr_median

(np.float64(0.8984064066604436), np.float64(0.9671075655431567))

In [25]:
pert_delt_pcorr_mean = np.mean(per_pert_pr_delt)
pert_delt_pcorr_median = np.median(per_pert_pr_delt)
pert_delt_pcorr_mean, pert_delt_pcorr_median

(np.float64(0.6067409307644542), np.float64(0.9619447843237646))

**Top 50 DEGs**

For our top 50 DEGs, we'll again calculate our correlation on only the top 50 differentially expressed genes (in our example the top 3). We'll calculate on the expression deltas as we're focused on differentially expressed genes.

Similar to our all genes, you can see how our last perturbation is positive while the $R^2$ was negative. We can also see our first two improved further by only focusing on the top expression in an excellent prediction. Finally our third is now fully negative showing major issues.

In [26]:
per_pert_pr_delt_topk = []
for i in range(unique_perts):
    print(f'----Pert {i}----')
    mp_r, mp_p  = mean_pert_real_delta[i], mean_pert_pred_delta[i]
    tk_idx = np.argsort(np.abs(mean_pert_real_delta[i]))[-TOP_K:]
    print(f'Top {TOP_K} expression deltas: real {mp_r[tk_idx]} | pred {mp_p[tk_idx]}')
    
    topk_corr, _ = pearsonr(mp_p[tk_idx], mp_r[tk_idx])
    topk_pcorr = 0.0 if np.isnan(topk_corr) else float(topk_corr)
    print(topk_pcorr)
    per_pert_pr_delt_topk.append(topk_pcorr)
    
per_pert_pr_delt_topk = np.array(per_pert_pr_delt_topk)
per_pert_pr_delt_topk.shape, per_pert_pr_delt_topk

----Pert 0----
Top 3 expression deltas: real [ 1.5 -1.8  2. ] | pred [ 1.45 -1.6   1.9 ]
0.9999956872672138
----Pert 1----
Top 3 expression deltas: real [ 0.1 -0.1 -0.1] | pred [ 0.1   -0.075 -0.075]
1.0
----Pert 2----
Top 3 expression deltas: real [-0.8 -1.   1.2] | pred [ 0.5  0.6 -0.5]
-1.0
----Pert 3----
Top 3 expression deltas: real [0.4 0.5 0.6] | pred [1.2 1.5 1.8]
1.0


((4,), array([ 0.99999569,  1.        , -1.        ,  1.        ]))

In [27]:
pert_delta_pcorr_topk_mean = np.mean(per_pert_pr_delt_topk)
pert_delta_pcorr_topk_median = np.median(per_pert_pr_delt_topk)
pert_delta_pcorr_topk_mean, pert_delta_pcorr_topk_median

(np.float64(0.49999892181680344), np.float64(0.9999978436336069))

## Cross-Perturbation Metrics

The metrics above evaluate each perturbation in isolation. The next set looks across all perturbations simultaneously to identify if the model can distinguish one perturbation's effect from another. We'll use the same data we prepared for the per-perturbation evals but run formulas across the data instead of per perturbation.

With the perturbation values, we will do a few different calculations: a centroid analysis, a control comparison, a review if we've correctly identified the major perturbation impacts, and finally a binned error analysis.

### Centroid Accuracy

For centroid accuracy, we are measuring if the predicted expression deltas are closest to the correct deltas, or is the distribution more like a different perturbation's expression deltas. We calculate this as follows:
$$\text{Centroid Accuracy} = \frac{1}{P}\sum_{p=1}^{P} \mathbb{1}\left[\underset{q}{\arg\min} | \hat{\delta}_p - \delta_q |^2 = p\right]$$

The centroid accuracy is the fraction of perturbations where the nearest match is the correct one. As we have more perturbations, including different perturbations with the same targets and modalities, this can become a challenging metric to increase. 

If we use just our perturbations based on how we've defined them, you can see that there is a potential issue that makes this challenge hard: if two perturbations have a different sequence but the same target, it can be very hard to tell them apart, and, arguably our model shouldn't be able to. We fix this by grouping our perturbation data by Target, Mode, and Cell type, taking the mean, and then comparing this new grouping. *Note that for our perturbation with no target, we use the Sequence*. 

In [28]:
pred_matrix = np.array([mean_pert_pred_delta[k] for k in range(unique_perts)])
pred_matrix.shape, pred_matrix

((4, 8),
 array([[ 0.7  , -1.1  ,  0.4  ,  1.9  , -0.4  ,  1.45 , -0.2  , -1.6  ],
        [ 0.1  , -0.075,  0.   , -0.075,  0.075,  0.05 , -0.025,  0.075],
        [-0.3  ,  0.5  ,  0.5  ,  0.6  , -0.2  ,  0.8  ,  0.3  , -0.5  ],
        [ 0.9  , -1.2  ,  0.6  ,  1.5  , -0.9  ,  1.8  , -0.3  ,  1.2  ]]))

In [29]:
real_matrix = np.array([mean_pert_real_delta[k] for k in range(unique_perts)])
real_matrix.shape, real_matrix

((4, 8),
 array([[ 0.8 , -1.25,  0.3 ,  2.  , -0.5 ,  1.5 , -0.1 , -1.8 ],
        [ 0.1 , -0.1 ,  0.05, -0.1 ,  0.05,  0.05,  0.  ,  0.05],
        [ 0.5 , -0.8 ,  0.6 , -1.  ,  0.3 ,  0.7 , -0.4 ,  1.2 ],
        [ 0.3 , -0.4 ,  0.2 ,  0.5 , -0.3 ,  0.6 , -0.1 ,  0.4 ]]))

**Pairwise Calculation**

Since we have a manageable amount of perturbations, we compute the full pairwise distance matrix $\|a - b\|^2 = \|a\|^2 + \|b\|^2 - 2 \, a \cdot b$. This avoids an explicit loop over perturbation pairs.

In [30]:
pred_sq = np.sum(pred_matrix**2, axis=1)
pred_sq = pred_sq[:, None]
pred_sq.shape, pred_sq

((4, 1),
 array([[10.3325  ],
        [ 0.035625],
        [ 1.97    ],
        [10.44    ]]))

In [31]:
real_sq = np.sum(real_matrix**2, axis=1)
real_sq = real_sq[None, :]
real_sq.shape, real_sq

((1, 4), array([[12.0425,  0.04  ,  4.43  ,  1.16  ]]))

In [32]:
two_ab = 2.0 * pred_matrix @ real_matrix.T
two_ab

array([[22.26  , -0.035 , -2.75  ,  4.1   ],
       [-0.1425,  0.07  ,  0.685 ,  0.125 ],
       [ 5.31  , -0.22  , -2.14  ,  0.84  ],
       [12.84  ,  0.39  ,  5.64  ,  6.96  ]])

**Pairwise Distance**

Now we'll calculate the pairwise distance with our components. The output of this will be $[\text{n\_perts}, \text{n\_perts}]$ where each row represents our predicted perturbation profile and each column is a distance to a real perturbation profile.

In [33]:
dist_matrix = pred_sq + real_sq - two_ab
dist_matrix.shape, dist_matrix

((4, 4),
 array([[1.1500000e-01, 1.0407500e+01, 1.7512500e+01, 7.3925000e+00],
        [1.2220625e+01, 5.6250000e-03, 3.7806250e+00, 1.0706250e+00],
        [8.7025000e+00, 2.2300000e+00, 8.5400000e+00, 2.2900000e+00],
        [9.6425000e+00, 1.0090000e+01, 9.2300000e+00, 4.6400000e+00]]))

**Centroid Accuracy**

Now that we have the pairwise distance, we now have to select for each perturbation which distribution is closest. We use a function that looks across each row and picks the index of the lowest value out, indicating the profile that matches closest.

You'll notice that our third perturbation, the one we purposefully made to be anti-correlated, actually matches closest to the real expression deltas of our second perturbation. This will impact our centroid accuracy since only 3/4 are correct.

In [34]:
closest_profile = np.argmin(dist_matrix, axis=1)
closest_profile.shape, closest_profile

((4,), array([0, 1, 1, 3]))

In [35]:
centroid_acc = float(np.mean(closest_profile == np.arange(unique_perts)))
centroid_acc

0.75

### Vs Baseline

The next metric is our "versus baseline" which can also be thought of as "compared to control". With this eval we look to see if the absolute predicted expression correlates better with the real perturbed expression than the control state. We'll use Pearson's $r$ since it is a measure of linear correlation. Given the amount of genes we use, this is especially challenging of a metric since most genes barely move meaning the control's correlation is very high with the perturbed state.

With this metric we calculate simply the "beat rate" or the amount of times that our prediction is better correlated than the control. For our testing we seeded a few different scenarios. You'll see that while our first two perturbations can beat the control, the last two cannot.

In [36]:
n_beat, n_eval_baseline = 0, 0

In [37]:
for key in range(unique_perts):
    print(f'---- Pert {key} ----')
    real_abs = mean_pert_real_abs[key]
    pred_abs = mean_pert_pred_abs[key]
    control = mean_pert_real_control[key]
    
    r_model, _ = pearsonr(pred_abs, real_abs)
    r_baseline, _ = pearsonr(control, real_abs)
    n_eval_baseline += 1
    winner = 'baseline'
    if r_model > r_baseline:
        n_beat += 1
        winner = 'model'
    print(f'Model Corr: {r_model} | Baseline Corr: {r_baseline} | Winner: {winner}')

---- Pert 0 ----
Model Corr: 0.997385112584632 | Baseline Corr: 0.6076356224861029 | Winner: model
---- Pert 1 ----
Model Corr: 0.9999286002882068 | Baseline Corr: 0.9983815285569038 | Winner: model
---- Pert 2 ----
Model Corr: 0.6594818952672543 | Baseline Corr: 0.8296747121948607 | Winner: baseline
---- Pert 3 ----
Model Corr: 0.9368300185016813 | Baseline Corr: 0.9642284068425429 | Winner: baseline


In [38]:
beat_rate = float(n_beat / n_eval_baseline)
beat_rate

0.5

### Severity Correlation

We next will measure the perturbation predicted severity to evaluate if our model correctly identifies which perturbations have larger and smaller effects. Severity is the overall strength of a perturbation's effect, measured as the L2 norm of the expression delta. We calculate severity as:

$$\text{severity}_i = \|\delta_i\|_2 = \sqrt{\sum_{g=1}^{G} \delta_{i,g}^2}$$

We compute this for both predicted and real expression deltas, then check whether the ranking of perturbations by severity is preserved. We'll calculate two different correlations:
1. Pearson's to capture the linear relationship
2. Spearman's to capture the rank ordering

**L2 Norm**
We'll start by calculating the L2 norm. This norm calculates the straight-line distance from the origin to a point in multidimensional space based on the formula above. This value is calculated per perturbation across all the genes (that act as the point).

In [39]:
pred_severity = np.array([np.linalg.norm(mean_pert_pred_delta[k]) for k in range(unique_perts)])
pred_severity

array([3.21442063, 0.18874586, 1.40356688, 3.23109888])

In [40]:
real_severity = np.array([np.linalg.norm(mean_pert_real_delta[k]) for k in range(unique_perts)])
real_severity

array([3.47023054, 0.2       , 2.10475652, 1.07703296])

**Pearson's $r$**

Now that we've calculated the L2 norm, we'll calculate the linear correlation of severity.

In [41]:
severity_pearson, _ = pearsonr(pred_severity, real_severity)
severity_pearson


np.float64(0.6151718784472734)

**Spearman's $r$**

Now that we've calculated the L2 norm, we'll calculate the rank order correlation of severity. We do this using a new correlation called Spearman's.

$$\rho_{\text{severity}} = 1 - \frac{6\sum_{p=1}^{P}(R(\hat{s}_p) - R(s_p))^{2}}{P(P^{2}-1)}$$

where $R(\cdot)$ denotes the rank of each value. The rank is the position of each value when sorted from smallest to largest. With Spearman's correlation we can determine whether the predicted severities have the same ordering, regardless of the actual magnitudes. We'll see that because of how we placed our predicted vs real values, we won't be quite right.

In [42]:
severity_spearman, _ = spearmanr(pred_severity, real_severity)
severity_spearman

np.float64(0.39999999999999997)

### Error by Magnitude

The final cross-perturbation metric is an error analysis based on the expected expression delta to help us troubleshoot prediction errors. Binning by real delta magnitude lets us see if the model's errors are uniform or if it struggles more on genes with large versus small expression changes. We use mean absolute error to keep the error in the same units as the expression values, making the per-bin numbers directly interpretable. MAE is calculated as:

$$\text{MAE}_\text{bin} = \frac{1}{|B_\text{bin}|}\sum_{(p,g) \in B_\text{bin}} |\hat{\delta}_{p,g} - \delta_{p,g}|$$

where $B_\text{bin}$ is the set of (perturbation, gene) pairs whose $|\delta_{p,g}|$ falls in the bin.

Because of how this is structured and calculated, we expect that the lower bins will have smaller error. This is because genes with small real deltas have less room to be wrong. If the real delta is 0.05, the error can't realistically be much larger than that unless the model is wildly off. For large real deltas, there's more room for the prediction to miss. So MAE naturally scales with magnitude, and the interesting signal is whether it scales proportionally or disproportionately.

We'll start by creating our bins structure and preparing our data. For this eval we concatenate all of our gene predictions into one long array.

In [43]:
magnitude_bins = [0, 0.25, 0.5, 1.0, 1.5, 2.0, np.inf]
bin_labels = ['0-0.25', '0.25-0.5', '0.5-1.0', '1.0-1.5', '1.5-2.0', '2.0+']
error_by_magnitude = {}

In [44]:
all_pred = np.concatenate([mean_pert_pred_delta[k] for k in range(unique_perts)])
all_pred.shape, all_pred

((32,),
 array([ 0.7  , -1.1  ,  0.4  ,  1.9  , -0.4  ,  1.45 , -0.2  , -1.6  ,
         0.1  , -0.075,  0.   , -0.075,  0.075,  0.05 , -0.025,  0.075,
        -0.3  ,  0.5  ,  0.5  ,  0.6  , -0.2  ,  0.8  ,  0.3  , -0.5  ,
         0.9  , -1.2  ,  0.6  ,  1.5  , -0.9  ,  1.8  , -0.3  ,  1.2  ]))

In [45]:
all_real = np.concatenate([mean_pert_real_delta[k] for k in range(unique_perts)])
all_real.shape, all_real

((32,),
 array([ 0.8 , -1.25,  0.3 ,  2.  , -0.5 ,  1.5 , -0.1 , -1.8 ,  0.1 ,
        -0.1 ,  0.05, -0.1 ,  0.05,  0.05,  0.  ,  0.05,  0.5 , -0.8 ,
         0.6 , -1.  ,  0.3 ,  0.7 , -0.4 ,  1.2 ,  0.3 , -0.4 ,  0.2 ,
         0.5 , -0.3 ,  0.6 , -0.1 ,  0.4 ]))

**Binned MAE**

Now that we have our expression, we need to calculate the mean absolute error. We'll first start by calculating the error and bin value, after which we'll slot the error into the bins.

In [46]:
all_errors, all_magnitudes = all_pred - all_real, np.abs(all_real)
all_errors, all_magnitudes

(array([-1.0000000e-01,  1.5000000e-01,  1.0000000e-01, -1.0000000e-01,
         1.0000000e-01, -5.0000000e-02, -1.0000000e-01,  2.0000000e-01,
         0.0000000e+00,  2.5000000e-02, -5.0000000e-02,  2.5000000e-02,
         2.5000000e-02,  4.4408921e-16, -2.5000000e-02,  2.5000000e-02,
        -8.0000000e-01,  1.3000000e+00, -1.0000000e-01,  1.6000000e+00,
        -5.0000000e-01,  1.0000000e-01,  7.0000000e-01, -1.7000000e+00,
         6.0000000e-01, -8.0000000e-01,  4.0000000e-01,  1.0000000e+00,
        -6.0000000e-01,  1.2000000e+00, -2.0000000e-01,  8.0000000e-01]),
 array([0.8 , 1.25, 0.3 , 2.  , 0.5 , 1.5 , 0.1 , 1.8 , 0.1 , 0.1 , 0.05,
        0.1 , 0.05, 0.05, 0.  , 0.05, 0.5 , 0.8 , 0.6 , 1.  , 0.3 , 0.7 ,
        0.4 , 1.2 , 0.3 , 0.4 , 0.2 , 0.5 , 0.3 , 0.6 , 0.1 , 0.4 ]))

In [47]:
for i in range(len(magnitude_bins) - 1):
    print(f'Bin {i}:{magnitude_bins[i]}')
    # create a mask for the bin where value >= bin value, and < next bin 
    mask = (all_magnitudes >= magnitude_bins[i]) & (all_magnitudes < magnitude_bins[i + 1])
    print(mask)
    if mask.sum() > 0:
        error_by_magnitude[bin_labels[i]] = {'mae': float(np.mean(np.abs(all_errors[mask]))), 'count': int(mask.sum())}

Bin 0:0
[False False False False False False  True False  True  True  True  True
  True  True  True  True False False False False False False False False
 False False  True False False False  True False]
Bin 1:0.25
[False False  True False False False False False False False False False
 False False False False False False False False  True False  True False
  True  True False False  True False False  True]
Bin 2:0.5
[ True False False False  True False False False False False False False
 False False False False  True  True  True False False  True False False
 False False False  True False  True False False]
Bin 3:1.0
[False  True False False False False False False False False False False
 False False False False False False False  True False False False  True
 False False False False False False False False]
Bin 4:1.5
[False False False False False  True False  True False False False False
 False False False False False False False False False False False False
 False False False Fa

In [48]:
error_by_magnitude

{'0-0.25': {'mae': 0.0795454545454545, 'count': 11},
 '0.25-0.5': {'mae': 0.5857142857142854, 'count': 7},
 '0.5-1.0': {'mae': 0.5874999999999999, 'count': 8},
 '1.0-1.5': {'mae': 1.1500000000000001, 'count': 3},
 '1.5-2.0': {'mae': 0.12500000000000022, 'count': 2},
 '2.0+': {'mae': 0.10000000000000009, 'count': 1}}

## Expression Prediction Final Wrapup
We've now walked through our evaluation for expression level prediction. As you can see we run a number of different evaluations that help tell us where our prediction is working well and where it is not. As shown, we run these evaluations on the total dataset, split up by perturbation, and across perturbation. In our production eval we also run 2 other subgroupings of the exact same calculation but with further subgroupings: by dataset, and by cell type. In the eval report you see a final focus on "GEARS" datasets which is just copying the per-dataset values but ones that we can directly compare to GEARS model performance.